In [ ]:
# Notebook parameters — edit these with your Kaggle slugs
# `dataset_slug` should be a Kaggle dataset slug like 'owner/dataset-name'
# `model_dataset_slug` is optional and can point to a dataset that contains model artifacts
dataset_slug = 'kausikvaibhavpatra/kaggle-rag-v1-code'  # <- change this to your dataset
model_dataset_slug = None  # <- optionally set to 'yourname/your-model-dataset'
model_path_within_dataset = ''  # e.g. 'models/best_model.pt'
auto_run_experiment = False  # set True to run experiment automatically after setup


In [ ]:
# Unpack dataset files into /kaggle/working using either an attached dataset
# or by downloading the Kaggle dataset specified in `dataset_slug`.
import os, tarfile, shutil, subprocess
from pathlib import Path

# `dataset_slug` and `model_dataset_slug` may be set in the parameters cell above.
# Examples: dataset_slug = 'kausikvaibhavpatra/kaggle-rag-v1-code'
dataset_slug = globals().get('dataset_slug', None)
model_dataset_slug = globals().get('model_dataset_slug', None)

# If a dataset slug was provided, download and unzip it into working/input_dataset
if dataset_slug:
    os.makedirs('/kaggle/working/input_dataset', exist_ok=True)
    print('Downloading dataset:', dataset_slug)
    cmd = ['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', '/kaggle/working/input_dataset', '--unzip']
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout or res.stderr)
    ds_name = dataset_slug.split('/')[-1]
    candidate = Path('/kaggle/working/input_dataset') / ds_name
    if candidate.exists():
        input_root = candidate
    else:
        input_root = Path('/kaggle/working/input_dataset')
else:
    input_root = Path('/kaggle/input')
    # Try to locate a matching attached dataset containing project files
    if not (input_root / 'kausikvaibhavpatra').exists():
        candidates = [p for p in Path('/kaggle/input').iterdir() if p.is_dir()]
        for cand in candidates:
            if (cand / 'run_experiment.py').exists() or (cand / 'src.tar').exists():
                input_root = cand
                break

# Optionally download a separate model dataset and expose `model_root`
if model_dataset_slug:
    os.makedirs('/kaggle/working/input_model', exist_ok=True)
    print('Downloading model dataset:', model_dataset_slug)
    cmd = ['kaggle', 'datasets', 'download', '-d', model_dataset_slug, '-p', '/kaggle/working/input_model', '--unzip']
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout or res.stderr)
    model_root = Path('/kaggle/working/input_model')
else:
    model_root = None

print('Using dataset root:', input_root)
print('Using model root:', model_root)
os.chdir('/kaggle/working')

# Copy root-level files from the dataset into the working dir
for name in ['run_experiment.py', 'config.yaml', 'requirements.txt', 'requirements-py311.txt', 'README.md']:
    src = input_root / name
    if src.exists():
        shutil.copy2(src, Path('/kaggle/working') / name)

# Copy source tree if it is present inside the dataset
for folder in ['src', 'scripts']:
    src_dir = input_root / folder
    dst_dir = Path('/kaggle/working') / folder
    if src_dir.exists() and src_dir.is_dir():
        if dst_dir.exists():
            shutil.rmtree(dst_dir)
        shutil.copytree(src_dir, dst_dir)

# Extract any tar archives that hold folders (src.tar, scripts.tar)
for tar_name in ['src.tar', 'scripts.tar']:
    tar_path = input_root / tar_name
    if tar_path.exists():
        with tarfile.open(tar_path) as tf:
            tf.extractall('/kaggle/working')

# If a model dataset was provided and contains files we want to surface, copy them
if model_root is not None:
    for p in model_root.iterdir():
        try:
            if p.is_file():
                shutil.copy2(p, Path('/kaggle/working') / p.name)
        except Exception:
            continue

print('Working files:', sorted([p.name for p in Path('/kaggle/working').iterdir() if p.is_file()])[:50])
print('src exists:', Path('/kaggle/working/src').exists())
print('scripts exists:', Path('/kaggle/working/scripts').exists())
print('pyproject exists:', Path('/kaggle/working/pyproject.toml').exists())
print('setup.cfg exists:', Path('/kaggle/working/setup.cfg').exists())


In [ ]:
# Install the project into the notebook environment so `import src` works reliably.
import sys, os, subprocess
from pathlib import Path
wd = '/kaggle/working'
os.chdir(wd)
print('Installing local project from:', wd)
cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-e', wd]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print('pip return code:', result.returncode)
if result.stdout:
    print(result.stdout[:2000])
if result.stderr:
    print(result.stderr[:2000])
if result.returncode != 0:
    raise RuntimeError('pip install -e /kaggle/working failed')
# quick import check after install
try:
    import src
    print('Imported src OK ->', getattr(src, '__file__', 'no __file__'))
except Exception as e:
    print('src import failed:', e)


In [ ]:
import sys, platform, os
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('CWD:', os.getcwd())
try:
    import torch
    print('Torch available, CUDA:', torch.cuda.is_available())
except Exception as e:
    print('Torch not available:', e)


In [ ]:
# Run experiment with an in-process bootstrap so PYTHONPATH is set for imports.
import sys, os, runpy
wd = '/kaggle/working'
if wd not in sys.path:
    sys.path.insert(0, wd)
srcp = os.path.join(wd, 'src')
if srcp not in sys.path:
    sys.path.insert(0, srcp)
os.environ['PYTHONPATH'] = wd + (os.pathsep + os.environ.get('PYTHONPATH','') if os.environ.get('PYTHONPATH') else '')
print('Running run_experiment.py (mode=smoke) with bootstrapped PYTHONPATH')
import argparse
# ensure argv forwarded to the script
import sys as _sys
_sys.argv = ['run_experiment.py', '--mode', 'smoke']
try:
    runpy.run_path('run_experiment.py', run_name='__main__')
except SystemExit as e:
    print('run_experiment exited with', e)
except Exception as e:
    import traceback
    traceback.print_exc()


In [ ]:
import json
from pathlib import Path
import pandas as pd
pred_path = Path('outputs/predictions/predictions.csv')
metrics_path = Path('outputs/metrics/metrics.json')
if pred_path.exists():
    display(pd.read_csv(pred_path).head())
code
#VSC-bootstrap-0001
python
# Bootstrap PYTHONPATH and ensure `src/` is present in /kaggle/working.
import sys, os, shutil, tarfile
from pathlib import Path
wd = '/kaggle/working'
if wd not in sys.path:
    sys.path.insert(0, wd)
srcp = os.path.join(wd, 'src')
if srcp not in sys.path:
    sys.path.insert(0, srcp)
os.environ['PYTHONPATH'] = wd + (os.pathsep + os.environ.get('PYTHONPATH','') if os.environ.get('PYTHONPATH') else '')
# If src/ was not included in the dataset, try to locate it under /kaggle/input/* and copy or extract it.
inp_root = Path('/kaggle/input')
if not Path(srcp).exists():
    for cand in inp_root.iterdir():
        try:
            cand = Path(cand)
            # direct folder named src
            if (cand / 'src').is_dir():
                shutil.copytree(cand / 'src', Path(srcp))
                break
            # archived src.tar
            if (cand / 'src.tar').is_file():
                with tarfile.open(cand / 'src.tar') as tf:
                    tf.extractall(wd)
                break
            # sometimes repo is at top level with src/
            if (cand / 'run_experiment.py').exists() and (cand / 'src').is_dir():
                shutil.copytree(cand / 'src', Path(srcp))
                break
        except Exception:
            continue
# report status
print('Working files:', sorted([p.name for p in Path(wd).iterdir() if p.is_file()])[:50])
print('src exists:', Path(srcp).exists())
print('PYTHONPATH:', os.environ.get('PYTHONPATH'))
# quick import check
try:
    import src
    print('Imported src OK ->', getattr(src, '__file__', 'no __file__'))
except Exception as e:
    print('src import failed:', e)
